In [1]:
import re
import string
import requests
from collections import Counter
import os
import sys
%pip install torch diffusers

import torch
import nltk

# Ensure NLTK data directory is configured and created
nltk_data_dir = os.path.join(os.path.expanduser("~"), "nltk_data")
os.makedirs(nltk_data_dir, exist_ok=True)

if nltk_data_dir not in nltk.data.path:
    nltk.data.path.insert(0, nltk_data_dir)

print(f"NLTK data directory: {nltk_data_dir}")
print(f"NLTK data paths: {nltk.data.path}")

# Download required NLTK data with error handling
try:
    print("\nDownloading 'punkt_tab' tokenizer...")
    nltk.download("punkt_tab", download_dir=nltk_data_dir, quiet=False)
    print("'punkt_tab' downloaded successfully")
except Exception as e:
    print(f"Error downloading 'punkt_tab': {e}")

try:
    print("\nDownloading 'stopwords'...")
    nltk.download("stopwords", download_dir=nltk_data_dir, quiet=False)
    print("'stopwords' downloaded successfully")
except Exception as e:
    print(f"Error downloading 'stopwords': {e}")

# Import NLTK components
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Verify downloads
print("\nVerifying NLTK data...")
try:
    # Test tokenization to ensure punkt_tab is working
    test_tokenize = word_tokenize("Test sentence")
    print(f"✓ NLTK tokenizer working: {test_tokenize}")
except Exception as e:
    print(f"✗ NLTK verification failed: {e}")

from rake_nltk import Rake

from transformers import pipeline

from PIL import Image
from io import BytesIO



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
NLTK data directory: C:\Users\vober\nltk_data
NLTK data paths: ['C:\\Users\\vober\\nltk_data', 'C:\\Users\\vober/nltk_data', 'c:\\Users\\vober\\OneDrive\\Documents\\ArtML\\venv\\nltk_data', 'c:\\Users\\vober\\OneDrive\\Documents\\ArtML\\venv\\share\\nltk_data', 'c:\\Users\\vober\\OneDrive\\Documents\\ArtML\\venv\\lib\\nltk_data', 'C:\\Users\\vober\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data']



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vober\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vober\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'punkt_tab' downloaded successfully

'stopwords' downloaded successfully

Verifying NLTK data...
✓ NLTK tokenizer working: ['Test', 'sentence']


c:\Users\vober\OneDrive\Documents\ArtML\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess_lyrics(text: str) -> list[str]: 
    """
    Cleans and tokenizes raw lyric text by removing punctuation, special 
    characters, and stopwords, preparing it for downstream NLP processing.
    """

    lowercase_text = text.lower()

    regex_rules = [
        (r"\[.*?\]", ""),
        (r"\(.*?\)", ""),
        (r"©|™|℗", ""),
        (r"https?://\S+|www\.\S+", ""),
        (r"^[-–—\s]+|[-–—\s]+$", ""),
        (r"\s{2,}", " "),
    ]

    for pattern, replacement in regex_rules:
        lowercase_text = re.sub(pattern, replacement, lowercase_text)

    tokenized_list = word_tokenize(lowercase_text)

    tokenized_list = [token for token in tokenized_list if token not in string.punctuation]

    english_stop_words = stopwords.words("english")

    filtered_words = [word for word in tokenized_list if word not in english_stop_words]

    return filtered_words


In [3]:
def extract_keywords(tokens: list[str]) -> list[str]:
    """
    Extracts the most thematically relevant keywords from preprocessed tokens
    to capture the core themes and subjects of the lyrics.
    """

    r = Rake(max_length=7)

    r.extract_keywords_from_text(" ".join(tokens))

    thematic_keywords = r.get_ranked_phrases()

    return thematic_keywords[:10]




    

In [4]:
def classify_emotions(text: str) -> list[tuple[str, float]]:
    """
    Runs emotion classification on the raw lyric text using a pretrained
    HuggingFace model (j-hartmann/emotion-english-distilroberta-base),
    returning a ranked list of emotions and their confidence scores.
    """

    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

    results = classifier(text)
    
    # Handle the results structure: if results is a list of lists (multiple sentences),
    # take the first one; if it's a list of dicts, use it directly
    if isinstance(results, list) and len(results) > 0:
        if isinstance(results[0], list):
            emotion_scores = results[0]
        else:
            emotion_scores = results
    else:
        emotion_scores = results
    
    emotions_with_scores = [(result['label'], result['score']) for result in emotion_scores]
    
    ranked_emotions = sorted(emotions_with_scores, key=lambda x: x[1], reverse=True)
    
    return ranked_emotions


In [5]:
def build_prompt(emotions: list[tuple[str, float]], keywords: list[str]) -> str:
    """
    Constructs a descriptive text prompt for image generation by combining
    the top detected emotions and thematic keywords extracted from the lyrics.
    """
    
    # Extract top emotions by confidence score
    top_emotions = emotions[:3]
    emotion_names = [emotion[0] for emotion in top_emotions]
    
    # Get top keywords
    top_keywords = keywords[:5]
    
    # Construct descriptive prompt
    emotion_str = ", ".join(emotion_names)
    keywords_str = ", ".join(top_keywords)
    
    prompt = f"An album cover art that conveys {emotion_str} emotions featuring {keywords_str}"
    
    return prompt

In [6]:
def generate_image(prompt: str) -> None:
    """
    Generates an album cover image using a fast, lightweight Stable Diffusion model.
    No API key needed. Uses optimized settings for speed.
    """
    
    try:
        from diffusers import StableDiffusionPipeline
        import torch
        import time
        
        print("Loading Stable Diffusion model (first run may take a moment)...")
        
        # Use a faster, lightweight model
        model_id = "runwayml/stable-diffusion-v1-5"
        
        # Check if GPU is available
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        
        # Initialize the pipeline with optimizations
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            safety_checker=None,  # Disable safety checker for faster generation
            requires_safety_checker=False
        )
        pipe = pipe.to(device)
        
        # Enable attention slicing for lower memory usage and faster inference
        pipe.enable_attention_slicing()
        
        print(f"\nGenerating image with prompt: {prompt}")
        print("This may take 1-3 minutes...")
        
        # Generate the image with optimized settings
        start_time = time.time()
        with torch.no_grad():
            image = pipe(
                prompt, 
                num_inference_steps=15,  # Reduced from 30 for speed
                guidance_scale=7.5,
                height=512,
                width=512
            ).images[0]
        
        elapsed_time = time.time() - start_time
        
        # Save the image
        image.save("album_cover.png")
        print(f"✓ Album cover saved as 'album_cover.png' (Generated in {elapsed_time:.1f} seconds)")
        
    except ImportError:
        print("Error: 'diffusers' package not installed. Installing now...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "diffusers"])
        print("Please run the cell again.")
    except Exception as e:
        print(f"Error generating image: {e}")


In [7]:
def main():
    import time
    
    # Example lyric text for testing
    sample_lyrics = """
    When the night falls and shadows creep
    I find myself lost in memories deep
    Every word you said, every tear I've cried
    In this darkness, I search for the light inside
    Feel the weight of the world on my shoulders now
    But I'll rise again, I'll figure it out somehow
    """
    
    total_start = time.time()
    
    # Step 1: Preprocess the lyrics
    print("=" * 60)
    print("STEP 1: Preprocessing lyrics...")
    step_start = time.time()
    preprocessed_tokens = preprocess_lyrics(sample_lyrics)
    print(f"Tokens: {preprocessed_tokens[:10]}...")
    print(f"✓ Completed in {time.time() - step_start:.2f}s")
    
    # Step 2: Extract keywords
    print("\n" + "=" * 60)
    print("STEP 2: Extracting keywords...")
    step_start = time.time()
    keywords = extract_keywords(preprocessed_tokens)
    print(f"Keywords: {keywords}")
    print(f"✓ Completed in {time.time() - step_start:.2f}s")
    
    # Step 3: Classify emotions
    print("\n" + "=" * 60)
    print("STEP 3: Classifying emotions...")
    step_start = time.time()
    emotions = classify_emotions(sample_lyrics)
    print(f"Emotions: {emotions}")
    print(f"✓ Completed in {time.time() - step_start:.2f}s")
    
    # Step 4: Build the prompt
    print("\n" + "=" * 60)
    print("STEP 4: Building prompt for image generation...")
    step_start = time.time()
    prompt = build_prompt(emotions, keywords)
    print(f"Generated prompt: {prompt}")
    print(f"✓ Completed in {time.time() - step_start:.2f}s")
    
    # Step 5: Generate the image
    print("\n" + "=" * 60)
    print("STEP 5: Generating image...")
    step_start = time.time()
    generate_image(prompt)
    print(f"✓ Completed in {time.time() - step_start:.2f}s")
    
    # Summary
    total_elapsed = time.time() - total_start
    print("\n" + "=" * 60)
    print(f"Total execution time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} minutes)")
    print("=" * 60)

# Call main to test the program
main()


STEP 1: Preprocessing lyrics...
Tokens: ['night', 'falls', 'shadows', 'creep', 'find', 'lost', 'memories', 'deep', 'every', 'word']...
✓ Completed in 0.01s

STEP 2: Extracting keywords...
Keywords: ['figure somehow', 'rise']
✓ Completed in 0.00s

STEP 3: Classifying emotions...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13810.47it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emotions: [('sadness', 0.6986406445503235)]
✓ Completed in 4.00s

STEP 4: Building prompt for image generation...
Generated prompt: An album cover art that conveys sadness emotions featuring figure somehow, rise
✓ Completed in 0.00s

STEP 5: Generating image...


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading Stable Diffusion model (first run may take a moment)...
Using device: cpu


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 3196.99it/s]4.19it/s]
CLIPTextModel LOAD REPORT from: C:\Users\vober\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.17it/s]



Generating image with prompt: An album cover art that conveys sadness emotions featuring figure somehow, rise
This may take 1-3 minutes...


100%|██████████| 15/15 [07:06<00:00, 28.43s/it]


✓ Album cover saved as 'album_cover.png' (Generated in 448.2 seconds)
✓ Completed in 453.35s

Total execution time: 457.4s (7.6 minutes)
